# 05-01 RunnablePassthrough: 데이터를 그대로 흘려보내기

Ch01에서 LCEL 파이프라인의 기본 구조를 배웠다면, 이제 파이프라인 안에서 **데이터를 그대로 통과시키거나 확장하는 방법**을 배워요.

체인을 구성하다 보면 "입력을 변환하지 않고 그대로 다음 단계로 넘기고 싶은" 상황이 생겨요. 예를 들어 RAG에서 사용자 질문은 검색기에도 보내야 하고, 동시에 프롬프트에도 그대로 전달해야 해요. 이때 사용하는 것이 `RunnablePassthrough`예요.

## 학습 목표

- `RunnablePassthrough()`로 입력을 변경하지 않고 그대로 다음 단계로 전달하는 방법을 설명할 수 있어요
- `RunnablePassthrough.assign()`으로 원본 데이터를 유지하면서 새로운 키를 동적으로 추가하는 방법을 구현할 수 있어요
- RAG(검색 증강 생성, Retrieval-Augmented Generation) 파이프라인에서 사용자 질문을 그대로 흘려보내는 패턴을 적용할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- LCEL(LangChain Expression Language) 파이프 연산자(`|`) 사용법 (Ch01 `03-LCEL_basic` 참고)
- `RunnableParallel`의 기본 동작 방식
- Python 딕셔너리와 람다(lambda) 함수

---

> 🔑 **핵심 개념**: `RunnablePassthrough`는 LCEL 파이프라인에서 데이터를 처리하는 두 가지 방식을 제공해요:
> - **그대로 전달**: 입력을 변경하지 않고 다음 단계로 전달
> - **키 추가**: 원본 데이터를 유지하면서 새로운 키-값 쌍을 동적으로 추가
>
> 이 클래스는 `Runnable` 인터페이스를 구현하므로, 다른 `Runnable` 객체와 함께 파이프라인에서 사용할 수 있어요.

아래 다이어그램은 `RunnablePassthrough`의 두 가지 모드를 한눈에 보여줘요:

```mermaid
flowchart LR
    subgraph mode1["모드 1: 그대로 전달"]
        direction LR
        A1["입력<br/>{'num': 1}"]:::input
        B1["RunnablePassthrough()"]:::process
        C1["출력<br/>{'num': 1}"]:::output
        A1 --> B1 --> C1
    end

    subgraph mode2["모드 2: 키 추가 (assign)"]
        direction LR
        A2["입력<br/>{'num': 1}"]:::input
        B2["assign(mult=lambda x: x['num']*3)"]:::process
        C2["출력<br/>{'num': 1, 'mult': 3}"]:::output
        A2 --> B2 --> C2
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API 키 정보 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")


## 1. RunnablePassthrough() - 입력 그대로 전달

`RunnablePassthrough()`를 단독으로 호출하면 입력을 변경하지 않고 그대로 전달해요.

일반적으로 `RunnableParallel`과 함께 사용되어 데이터를 맵의 새로운 키에 할당하는 데 활용돼요.

> 🔑 **핵심 개념**: `RunnablePassthrough()`는 데이터를 복사하거나 변환하지 않고 그대로 흘려보내는 "투명한 파이프(transparent pipe)"예요. 데이터가 통과하는 동안 아무런 가공도 일어나지 않아요. RAG에서 원본 질문을 보존할 때 필수적으로 사용돼요.

> 🎯 **강의 포인트**: 아래 코드에서 `passed`, `extra`, `modified` 세 가지 방식의 출력 차이를 비교하면 `RunnablePassthrough`의 역할이 명확하게 드러나요. 세 결과를 나란히 놓고 "원본 유지 vs 확장 vs 변환"의 차이를 설명하면 이해가 빨라요.

In [3]:
# ============================================================
# TODO: RunnableParallel과 RunnablePassthrough를 조합하여 체인을 완성하세요
# 힌트: passed=RunnablePassthrough(), extra=RunnablePassthrough.assign(...), modified=lambda ...
# 예상 결과: {'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}
# ============================================================

# ---------------------------------------------------
# RunnableParallel + RunnablePassthrough 세 가지 방식 비교
# ---------------------------------------------------

# 🎯 강의 포인트: 세 가지 방식(passed/extra/modified)의 출력 차이를 비교하면
#                RunnablePassthrough의 역할이 명확하게 드러납니다

# 1단계: RunnableParallel 생성
# - passed: 원본 입력을 그대로 전달 (RunnablePassthrough())
# - extra: 원본에 "mult" 키를 추가 (RunnablePassthrough.assign())
# - modified: 입력값을 직접 변환하는 람다 함수
runnable = RunnableParallel(
    # RunnablePassthrough(): 입력을 변경 없이 그대로 통과
    passed=RunnablePassthrough(),
    # assign(): 원본 딕셔너리를 유지하면서 새로운 키-값 쌍을 추가
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    # 람다 함수: 입력에서 특정 값을 꺼내 변환
    modified=lambda x: x["num"] + 1,
)

# 2단계: 실행 결과 확인
result = runnable.invoke({"num": 1})
print("=" * 60)
print("📥 실행 결과")
print("=" * 60)
print(result)

📥 실행 결과
{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}


### 결과 분석

- `passed`: 원본 입력 `{'num': 1}`이 그대로 전달돼요
- `extra`: 원본 입력에 `mult` 키가 추가되어 `{'num': 1, 'mult': 3}` 형태로 반환돼요
- `modified`: `num`에 1을 더한 단순 변환 값 `2`가 반환돼요

**핵심 정리:**

| 방식 | 역할 | 출력 예시 |
|------|------|-----------|
| `RunnablePassthrough()` | 입력 그대로 통과 | `{'num': 1}` |
| `RunnablePassthrough.assign()` | 원본 유지 + 키 추가 | `{'num': 1, 'mult': 3}` |
| 람다 함수 | 값만 변환 | `2` |

> 💡 **실무 팁**: `RunnablePassthrough.assign()`은 체인 중간에서 원본을 잃지 않고 부가 정보를 붙이고 싶을 때 사용해요. 예를 들어 타임스탬프나 캐시 키를 덧붙이는 경우가 전형적인 활용 사례예요.

> ⚠️ **주의**: `assign()`에 전달하는 람다 함수는 반드시 **딕셔너리 전체**를 인자로 받아요. `lambda x: x["num"]`처럼 키로 접근해야지, `lambda num: num`처럼 값을 직접 받으면 동작하지 않아요.

In [4]:
# RunnablePassthrough.assign() 단독 사용 예제
# 
# assign()만 사용하면 원본 입력에 새로운 키만 추가됨

r = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3)
result = r.invoke({"num": 1})

print("=" * 60)
print("📥 RunnablePassthrough.assign() 결과")
print("=" * 60)
print(f"입력: {{'num': 1}}")
print(f"출력: {result}")
print()
print("💡 원본 입력('num')은 유지되고, 'mult' 키만 추가됨")


📥 RunnablePassthrough.assign() 결과
입력: {'num': 1}
출력: {'num': 1, 'mult': 3}

💡 원본 입력('num')은 유지되고, 'mult' 키만 추가됨


## 2. RunnablePassthrough.assign() — 키를 추가하며 흘려보내기

`RunnablePassthrough.assign()`을 사용하면 원본 입력을 유지하면서 동적으로 새로운 키-값 쌍을 추가할 수 있어요.

**언제 사용하면 좋을까요?**

- 체인 중간에서 원본 데이터를 보존하면서 **계산된 값이나 메타데이터를 추가**해야 할 때
- 입력 데이터를 기반으로 **동적으로 새로운 정보를 생성**해야 할 때
- 예: 타임스탬프 추가, 조건부 값 계산, 외부 API 호출 결과 추가

> 🔑 **핵심 개념**: `assign()`은 입력 딕셔너리를 "복사한 뒤 새 키를 추가"하는 방식으로 동작해요. 원본 딕셔너리를 변경(mutate)하는 것이 아니라, 확장된 새 딕셔너리를 만들어서 반환하는 거예요.

아래 다이어그램은 `assign()`이 데이터를 어떻게 처리하는지 보여줘요:

```mermaid
flowchart LR
    A["입력<br/>{product_name, price, is_on_sale}"]:::input
    B["assign()<br/>discount_rate 계산<br/>current_time 추가"]:::process
    C["확장된 입력<br/>{product_name, price, is_on_sale,<br/>discount_rate, current_time}"]:::process
    D["ChatPromptTemplate"]:::process
    E["LLM"]:::process
    F["상품 설명 출력"]:::output

    A --> B --> C --> D --> E --> F

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [5]:
# ============================================================
# TODO: RunnablePassthrough.assign()으로 동적 필드를 추가하는 체인을 구성하세요
# 힌트: chain = (RunnablePassthrough.assign(key=lambda x: ...) | prompt | model | StrOutputParser())
# 예상 결과: 할인율(30%)과 현재 시간이 포함된 상품 설명 생성
# ============================================================

# ---------------------------------------------------
# RunnablePassthrough.assign()으로 원본 입력에 새로운 정보 추가하기
# ---------------------------------------------------

from datetime import datetime

# 1단계: 프롬프트 템플릿 정의
# - assign()이 추가할 discount_rate, current_time 변수를 템플릿에 포함
product_prompt = ChatPromptTemplate.from_template(
    """다음 상품 정보를 바탕으로 매력적인 상품 설명을 작성해주세요.

상품명: {product_name}
가격: {price}원
할인율: {discount_rate}%
현재 시간: {current_time}

상품 설명:"""
)

# 2단계: RunnablePassthrough.assign()으로 'discount_rate'와 'current_time' 키를 동적으로 추가
#        - 원본 입력(product_name, price)은 그대로 유지
#        - assign()이 새로운 키들을 계산하여 추가
chain = (
    RunnablePassthrough.assign(
        discount_rate=lambda x: 30 if x.get("is_on_sale") else 0,  # 할인 중이면 30%, 아니면 0%
        current_time=lambda x: datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")
    )
    | product_prompt
    | model
    | StrOutputParser()
)

# 3단계: 실행 - 원본 입력에 할인율과 현재시간이 자동으로 추가되어 전달됨
original_input = {
    "product_name": "무선 이어폰",
    "price": 50000,
    "is_on_sale": True
}

print("=" * 60)
print("📥 1단계: 원본 입력 데이터")
print("=" * 60)
print(f"원본 입력: {original_input}")
print()

result = chain.invoke(original_input)

print("=" * 60)
print("📤 2단계: LLM 생성 결과")
print("=" * 60)
print(result)
print()

print("💡 핵심 정리")
print("=" * 60)
print("• 원본 입력은 그대로 유지됨")
print("• RunnablePassthrough.assign()이 키를 추가")
print("• 원본 데이터는 손실되지 않고 모두 보존됨!")


📥 1단계: 원본 입력 데이터
원본 입력: {'product_name': '무선 이어폰', 'price': 50000, 'is_on_sale': True}

📤 2단계: LLM 생성 결과
**무선 이어폰 - 자유로운 사운드, 특별한 할인!**

가격: ~70,000원~ → **50,000원** (30% 할인)

음악과 함께하는 모든 순간을 특별하게 만들어 줄 **무선 이어폰**을 소개합니다! 

최신 블루투스 기술을 탑재한 이 이어폰은 뛰어난 음질과 안정적인 연결성을 자랑합니다. 어떤 환경에서도 선명한 소리를 제공하여, 집안일을 하거나 운동을 할 때에도 음악을 즐길 수 있습니다. 컴팩트한 디자인으로 휴대가 간편하며, 귀에 착 달라붙는 인체공학적 형태로 편안한 착용감을 제공합니다.

특히, 30% 할인된 50,000원에 만나볼 수 있는 이번 기회는 놓치지 마세요! 긴 배터리 수명으로 간편하게 하루 종일 사용 가능하며, 물리적인 버튼이 아닌 터치 컨트롤로 편리함을 더했습니다. 

지금 바로 이 매력적인 무선 이어폰으로 일상 속 음악의 자유를 만끽하세요! 

**구매는 간편하게, 할인은 빠르게!**

💡 핵심 정리
• 원본 입력은 그대로 유지됨
• RunnablePassthrough.assign()이 키를 추가
• 원본 데이터는 손실되지 않고 모두 보존됨!


## 3. RAG 패턴에서의 RunnablePassthrough

`assign()`으로 데이터를 확장하는 방법을 익혔으니, 이번에는 `RunnablePassthrough()`가 가장 자주 쓰이는 패턴인 RAG(Retrieval-Augmented Generation)를 살펴볼게요.

RAG에서는 사용자의 질문이 두 군데에서 동시에 필요해요:
1. **벡터 저장소(Vector Store)**에서 관련 문서를 검색하는 데
2. **최종 프롬프트**에서 LLM에게 전달하는 데

`RunnablePassthrough()`는 질문을 복사하지 않고도 두 경로에 동시에 흘려보내는 역할을 해요.

> 🎯 **강의 포인트**: RAG 파이프라인에서 `"question": RunnablePassthrough()`는 사실상 관용구(idiom)예요. 이 패턴이 왜 필요한지 --- 질문이 retriever와 프롬프트 양쪽 모두에 필요하기 때문 --- 를 설명하면 학생들이 금방 이해해요.

> 💡 **실무 팁**: `RunnableParallel` 딕셔너리 안에서 `retriever`와 `RunnablePassthrough()`를 나란히 배치하면, 검색과 질문 전달이 **동시에(parallel)** 실행돼요. 검색이 오래 걸려도 질문 전달은 즉시 완료되므로 전체 대기 시간이 줄어들어요.

```mermaid
flowchart TD
    Q["사용자 질문"]:::input
    R["Retriever<br/>문서 검색"]:::process
    P["RunnablePassthrough()<br/>질문 그대로 전달"]:::process
    C["context: 검색된 문서"]:::process
    QQ["question: 원본 질문"]:::process
    PT["ChatPromptTemplate"]:::process
    LLM["LLM"]:::process
    A["답변"]:::output

    Q --> R --> C --> PT
    Q --> P --> QQ --> PT
    PT --> LLM --> A

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [6]:
# ============================================================
# TODO: RAG 체인에서 RunnablePassthrough()로 질문을 전달하는 구조를 완성하세요
# 힌트: {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | model | ...
# 예상 결과: "김민수의 직업은 머신러닝 엔지니어입니다." 형태의 답변
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# ---------------------------------------------------
# RAG 체인에서 RunnablePassthrough()의 역할 이해하기
# ---------------------------------------------------

# 1단계: FAISS 벡터 저장소 생성 (예제 데이터: 회사 직원 정보)
# - 텍스트를 임베딩으로 변환하여 유사도 검색 가능한 저장소 구축
vectorstore = FAISS.from_texts(
    [
        "김민수는 AI 스타트업에서 머신러닝 엔지니어로 근무하고 있습니다.",
        "이지은은 같은 회사에서 프론트엔드 개발자로 일하고 있습니다.",
        "김민수의 주요 업무는 자연어 처리 모델 개발입니다.",
        "이지은의 주요 업무는 React 기반 웹 애플리케이션 개발입니다.",
        "두 사람은 협업하여 AI 기반 웹 서비스를 개발하고 있습니다.",
    ],
    embedding=OpenAIEmbeddings(),
)

# 2단계: 벡터 저장소를 검색기(Retriever)로 변환
# - as_retriever()는 VectorStore를 Runnable 인터페이스의 검색기로 바꿔줌
# - 이렇게 해야 LCEL 파이프라인(|)에서 바로 연결 가능
retriever = vectorstore.as_retriever()

# 3단계: RAG 프롬프트 템플릿 정의
# - {context}: retriever가 검색한 문서가 들어갈 자리
# - {question}: 사용자 질문이 그대로 들어갈 자리 (RunnablePassthrough가 담당)
template = """다음 컨텍스트를 바탕으로 질문에 답변해주세요.
컨텍스트에 없는 내용은 추측하지 마세요.

컨텍스트:
{context}

질문: {question}

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# 4단계: 검색된 문서를 하나의 문자열로 결합하는 함수
def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 결합"""
    return "\n".join([doc.page_content for doc in docs])

# 5단계: RAG 체인 구성
# - "context" 경로: retriever로 문서 검색 → format_docs로 텍스트 결합
# - "question" 경로: RunnablePassthrough()로 질문 문자열을 그대로 전달
# - 두 경로의 결과가 합쳐져서 prompt → model → parser로 이어짐
#
# ⚠️ 주의: invoke()에 문자열을 넘길 때만 이 패턴이 동작해요
#           딕셔너리가 아닌 문자열 입력이 RunnablePassthrough()를 통해
#           "question" 키의 값으로 배정됩니다
retrieval_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()  # 입력 문자열을 그대로 question 키로 전달
    }
    | prompt
    | model
    | StrOutputParser()
)

print("=" * 60)
print("✅ RAG 체인 구성 완료")
print("=" * 60)
print("체인 구조:")
print("  입력(질문) → RunnablePassthrough() → 프롬프트 → LLM → 출력")
print()

✅ RAG 체인 구성 완료
체인 구조:
  입력(질문) → RunnablePassthrough() → 프롬프트 → LLM → 출력



In [7]:
# RAG 체인 실행 예제 1
question1 = "김민수의 직업은 무엇입니까?"
answer1 = retrieval_chain.invoke(question1)

print("=" * 60)
print("📥 질문 1")
print("=" * 60)
print(f"질문: {question1}")
print()
print("📤 답변:")
print(answer1)


📥 질문 1
질문: 김민수의 직업은 무엇입니까?

📤 답변:
김민수의 직업은 머신러닝 엔지니어입니다.


In [8]:
# RAG 체인 실행 예제 2
question2 = "이지은의 주요 업무는 무엇입니까?"
answer2 = retrieval_chain.invoke(question2)

print("=" * 60)
print("📥 질문 2")
print("=" * 60)
print(f"질문: {question2}")
print()
print("📤 답변:")
print(answer2)


📥 질문 2
질문: 이지은의 주요 업무는 무엇입니까?

📤 답변:
이지은의 주요 업무는 React 기반 웹 애플리케이션 개발입니다.


## 정리

### RunnablePassthrough의 두 가지 사용법

| 사용법 | 역할 | 주요 사용 사례 |
|--------|------|----------------|
| `RunnablePassthrough()` | 입력을 변경 없이 그대로 전달 | RAG에서 질문 전달, 데이터 흐름 유지 |
| `RunnablePassthrough.assign()` | 원본 데이터 유지 + 새로운 키 추가 | 타임스탬프, 조건부 값, 메타데이터 추가 |

### 핵심 요점

- `RunnablePassthrough()`는 입력을 변경하지 않고 그대로 전달해요 --- "투명한 파이프" 역할이에요
- `RunnablePassthrough.assign()`은 원본 데이터 손실 없이 추가 정보를 전달해요
- RAG 시스템에서 `"question": RunnablePassthrough()` 패턴은 거의 관용구처럼 사용돼요
- `RunnableParallel`과 함께 사용하면 여러 경로로 데이터를 분기할 수 있어요

### 언제 무엇을 쓸까?

| 상황 | 선택 | 이유 |
|------|------|------|
| 입력을 다음 단계에 그대로 넘기고 싶을 때 | `RunnablePassthrough()` | 변환 없이 통과 |
| 원본 입력에 계산된 필드를 추가하고 싶을 때 | `RunnablePassthrough.assign()` | 원본 유지 + 확장 |
| 입력에서 특정 값만 꺼내 변환하고 싶을 때 | `lambda` 또는 `RunnableLambda` | 자유로운 변환 |

### 다음 노트북 예고

다음 노트북(`02-Inspect-Runnables.ipynb`)에서는 지금까지 구성한 체인의 내부 구조를 시각적으로 확인하는 방법을 배워요. `get_graph()`, `print_ascii()` 등 디버깅에 유용한 도구를 살펴볼게요.